# Deep Burst SR Colab Starter

This notebook mounts Google Drive, prepares a synthetic RGB dataset from DIV2K, trains the model, and writes checkpoints plus sample outputs to Drive.

In [49]:
import os
IN_COLAB = 'COLAB_GPU' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not running in Colab: skipping Google Drive mount.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
from pathlib import Path

if IN_COLAB:
    %cd /content
    REPO_ROOT = Path('/content/deep_burst_sr_reimplementation')
    if not REPO_ROOT.exists():
        !git clone https://github.com/Karan2005-sys/deep_burst_sr_reimplementation.git {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd()

%cd {REPO_ROOT}
!pip install -r {REPO_ROOT / 'requirements.txt'}
print('Using repo:', REPO_ROOT)


/content
fatal: destination path 'deep_burst_sr_reimplementation' already exists and is not an empty directory.
/content/deep_burst_sr_reimplementation
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.6
    Uninstalling datasets-2.14.6:
      Successfully uninstalled datasets-2.14.6


In [51]:
if IN_COLAB:
    SYNTH_ROOT = '/content/drive/MyDrive/SyntheticRGB'
    REAL_ROOT = '/content/drive/MyDrive/BurstSR'
    OUTPUT_ROOT = '/content/drive/MyDrive/dbsr_outputs'
else:
    SYNTH_ROOT = '/tmp/SyntheticRGB'
    REAL_ROOT = '/tmp/BurstSR'
    OUTPUT_ROOT = '/tmp/dbsr_outputs'

SYNTH_CONFIG = 'configs/synthetic_rgb.yaml'
REAL_CONFIG = 'configs/real_burstsr.yaml'

print('SYNTH_ROOT:', SYNTH_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)


In [52]:
# Keep datasets version aligned with requirements.txt
!python - <<'PY'
import datasets
print('datasets version:', datasets.__version__)
PY


## Prepare synthetic RGB data from DIV2K
This creates `train/` and `val/` under `SYNTH_ROOT` automatically using DIV2K HR images.

In [53]:
!mkdir -p {OUTPUT_ROOT}
!python {REPO_ROOT / 'scripts' / 'prepare_div2k_synthetic.py'} --output-root {SYNTH_ROOT}


Traceback (most recent call last):
  File "/content/deep_burst_sr_reimplementation/scripts/prepare_div2k_synthetic.py", line 7, in <module>
    from datasets import load_dataset
  File "/usr/local/lib/python3.12/dist-packages/datasets/__init__.py", line 22, in <module>
    from .arrow_dataset import Dataset
  File "/usr/local/lib/python3.12/dist-packages/datasets/arrow_dataset.py", line 67, in <module>
    from .arrow_writer import ArrowWriter, OptimizedTypedSequence
  File "/usr/local/lib/python3.12/dist-packages/datasets/arrow_writer.py", line 27, in <module>
    from .features import Features, Image, Value
  File "/usr/local/lib/python3.12/dist-packages/datasets/features/__init__.py", line 18, in <module>
    from .features import Array2D, Array3D, Array4D, Array5D, ClassLabel, Features, Sequence, Value
  File "/usr/local/lib/python3.12/dist-packages/datasets/features/features.py", line 634, in <module>
    class _ArrayXDExtensionType(pa.PyExtensionType):
                           

In [54]:
!python {REPO_ROOT / 'scripts' / 'train.py'} --config {REPO_ROOT / SYNTH_CONFIG} --data-root {SYNTH_ROOT} --output-dir {OUTPUT_ROOT}/synthetic_run


2026-04-16 15:39:46.549798: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-16 15:39:46.554975: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-16 15:39:46.572892: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776353986.597608    7983 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776353986.604513    7983 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776353986.622062    7983 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [55]:
!find {OUTPUT_ROOT}/synthetic_run -maxdepth 3 -type f | sort | tail -n 20

find: ‘/content/drive/MyDrive/dbsr_outputs/synthetic_run’: No such file or directory


In [56]:
from IPython.display import display
from PIL import Image
from pathlib import Path

visual_root = Path(OUTPUT_ROOT) / 'synthetic_run' / 'visuals'
epoch_dirs = sorted([p for p in visual_root.glob('epoch_*') if p.is_dir()])
latest = epoch_dirs[-1] if epoch_dirs else None
print('Latest preview folder:', latest)
if latest is not None:
    display(Image.open(latest / 'prediction.png'))
    display(Image.open(latest / 'target.png'))

Latest preview folder: None


## Later: switch to real BurstSR
When the real dataset is available, point `REAL_ROOT` to it and run the command below.

In [57]:
# !python {REPO_ROOT / 'scripts' / 'train.py'} --config {REPO_ROOT / REAL_CONFIG} --data-root {REAL_ROOT} --output-dir {OUTPUT_ROOT}/real_run
